# Interactive Hook Sessions (Offline)

`HookSession` is TDHook's imperative interface for temporary capture and intervention. This notebook presents its targets, result objects, lifecycle, workflow integration, and managed early stopping with a local model.

In [1]:
import torch
from tensordict import TensorDict
from torch import nn

from tdhook.session import HookSession
from tdhook.targets import Target

torch.manual_seed(0)


class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(4, 6)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(6, 2)

    def forward(self, input):
        return self.linear2(self.relu(self.linear1(input)))


model = TinyModel().eval()
inputs = torch.randn(3, 4)
data = TensorDict({"input": inputs}, batch_size=[len(inputs)])

## Targets and capture results

A `Target` names a model-relative module, a value kind, its feature axis, and selected indices. `capture` returns a mutable `CapturedTarget`: `value` is the latest detached observation, while `values` retains observations from every matching call in order.

In [2]:
activation = Target("linear1", "activation", -1, (0, 2))

with HookSession(model) as session:
    captured = session.capture(activation)
    first_output = model(inputs)
    second_output = model(inputs + 1)
    active_program = session.program

assert len(captured.values) == 2
assert captured.values[0].shape == (3, 2)
assert len(active_program.hooks) == 1
[tuple(value.shape) for value in captured.values]

[(3, 2), (3, 2)]

## Activation and parameter replacement

`replace` applies only while the context is active. Activation hooks are removed and parameter values are restored on exit, including exceptional exits.

In [3]:
baseline = model(inputs)
output_unit = Target("", "activation", -1, (0,))
parameter_row = Target("linear1", "parameter", 0, (0,), parameter="weight")
original_weight = model.linear1.weight.detach().clone()

with HookSession(model) as session:
    session.replace(output_unit, 0)
    session.replace(parameter_row, -1)
    changed = model(inputs)
    assert torch.equal(changed[:, 0], torch.zeros_like(changed[:, 0]))
    assert not torch.equal(model.linear1.weight, original_weight)

assert torch.equal(model.linear1.weight, original_weight)
torch.testing.assert_close(model(inputs), baseline)

## Live capture-to-replacement routing

Pass a `CapturedTarget` directly to `replace` to route a fresh value to a later compatible target during the same execution. An optional `transform` runs immediately before replacement. Captures are detached clones by default; use `detach=False` only when the replacement must preserve autograd history. The resulting `HookProgram` records the source hook index and detach choice. A destination reached before a fresh source capture fails instead of reusing a value from an earlier model call.

In [4]:
source_units = Target("linear1", "activation", -1, (0, 2))
destination_units = Target("relu", "activation", -1, (1, 3))

with HookSession(model) as session:
    live = session.capture(source_units, detach=False)
    session.replace(destination_units, live, transform=torch.abs)
    routed_output = model(inputs)

dependency = session.program.hooks[1].source
assert dependency is not None
assert dependency.hook_index == 0
assert dependency.detach is False
routed_output.shape

torch.Size([3, 2])

## Forward input operations

Pass `direction="fwd_pre"` to capture or replace positional forward inputs. `direction="fwd_pre_kwargs"` exposes the hook value as `(args, kwargs)`; use `output_path=(0, 0)` for the first positional argument or `output_path=(1, "scale")` for keyword `scale`. The selected tuple, list, mapping, or TensorDict structure is preserved.

In [5]:
input_unit = Target("linear1", "activation", -1, (0,), output_path=(0,))

with HookSession(model) as session:
    captured_input = session.capture(input_unit, direction="fwd_pre")
    session.replace(input_unit, -1, direction="fwd_pre")
    input_modified_output = model(inputs)

torch.testing.assert_close(captured_input.values[-1], inputs[:, :1])
assert input_modified_output.shape == (3, 2)

## Gradient operations

Gradient targets use the same capture and replacement interface. `direction="bwd"` selects gradient inputs and `direction="bwd_pre"` selects gradient outputs; `bwd_pre` remains the default for compatibility. Backward must run inside the session so its temporary backward hooks are still installed.

In [6]:
gradient_input = inputs.detach().clone().requires_grad_()
gradient_output = Target("linear1", "gradient", -1, (0,))
gradient_input_target = Target("linear1", "gradient", -1, (0,))

with HookSession(model) as session:
    captured_gradient_output = session.capture(gradient_output)
    captured_gradient_input = session.capture(gradient_input_target, direction="bwd")
    session.replace(gradient_input_target, 0, direction="bwd")
    model(gradient_input).sum().backward()

assert captured_gradient_output.values[-1].shape == (3, 1)
assert captured_gradient_input.values[-1].shape == (3, 1)
assert gradient_input.grad is not None
assert torch.equal(gradient_input.grad[:, 0], torch.zeros(3))

## Observe a declared workflow

Nest a `HookSession` around a workflow when interactive operations should observe every model execution. Workflow methods remain sequential and explicit.

In [7]:
from tdhook.latent import ActivationCaching
from tdhook.workflow import Workflow

workflow = Workflow(
    ActivationCaching("linear1", cache_key=("activations", "first")),
    ActivationCaching("linear2", cache_key=("activations", "second")),
)

with HookSession(model) as session:
    workflow_capture = session.capture(activation)
    execution = workflow(model, data.clone())

assert len(workflow_capture.values) == 2
execution

TensorDict(
    fields={
        activations: TensorDict(
            fields={
                first: TensorDict(
                    fields={
                        linear1: Tensor(shape=torch.Size([3, 6]), device=cpu, dtype=torch.float32, is_shared=False)},
                    batch_size=torch.Size([3]),
                    device=None,
                    is_shared=False),
                second: TensorDict(
                    fields={
                        linear2: Tensor(shape=torch.Size([3, 2]), device=cpu, dtype=torch.float32, is_shared=False)},
                    batch_size=torch.Size([3]),
                    device=None,
                    is_shared=False)},
            batch_size=torch.Size([3]),
            device=None,
            is_shared=False),
        input: Tensor(shape=torch.Size([3, 4]), device=cpu, dtype=torch.float32, is_shared=False),
        output: Tensor(shape=torch.Size([3, 2]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.S

## Managed early stopping

`stop` ends forward execution after a selected module and suppresses TDHook's internal control-flow signal at the session boundary. It returns an `EarlyStopResult` containing the exact partial output. Gradient operations cannot be combined with early stopping.

In [8]:
with HookSession(model) as session:
    stopped = session.stop("relu")
    workflow(model, data.clone())

assert stopped.reached
assert stopped.output is not None
assert stopped.output.shape == (3, 6)
assert session.program.stopped_at == "relu"
assert all(not module._forward_hooks for module in model.modules())
stopped.output

tensor([[0.0000, 0.4749, 0.3604, 0.0000, 0.0000, 0.8189],
        [0.0000, 0.0000, 0.4791, 0.4055, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.1737, 0.0000, 0.0000, 0.0000]],
       grad_fn=<ReluBackward0>)

## Select repeated module calls

A module instance can run more than once during one root model pass. Set `Target.occurrences` to the ordered, zero-based calls you need; leaving it as `None` selects every call. Indices reset for each root pass. `session.program.occurrence_plans` records the requested target path and indices, while `session.occurrence_evidence` records the selected and observed indices for every validated pass. Missing occurrences fail before a model result is returned, duplicate or reordered indices are rejected when the target is created, and all temporary hooks are removed on success or failure.

In [9]:
class SharedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared = nn.Identity()

    def forward(self, value):
        calls = (self.shared(value + offset) for offset in (1, 2, 3))
        return torch.cat(tuple(calls), dim=-1)


shared_model = SharedModel()
repeated = Target("shared", "activation", -1, (0,), occurrences=(0, 2))

with HookSession(shared_model) as session:
    selected_calls = session.capture(repeated)
    shared_model(torch.zeros(1, 1))

assert [value.item() for value in selected_calls.values] == [1.0, 3.0]
assert session.program.occurrence_plans[0].selected_indices == (0, 2)
assert session.occurrence_evidence[0].observed_indices == (0, 1, 2)
assert all(not module._forward_hooks for module in shared_model.modules())

## Choosing an interface

Use `method.prepare(model)` for one configured method, `Workflow` for sequential composition, and `HookSession` for interactive operations whose lifecycle you control directly. Nest a session around a workflow when the operations should observe each workflow model call.